In [1]:
"""
==============================================================================
🚀 [Step 2] Data Stratification & Copy-Paste Augmentation
==============================================================================
### @Author       : 한의정
### @Description  : 계층적 분할(Stratified Split)을 통한 독립적 Val 셋 구축 및
###                 희귀 클래스 물량 확보를 위한 지능형 합성(Copy-Paste) 수행.
==============================================================================
"""

'\n==============================================================================\n🚀 [Step 2] Data Stratification & Copy-Paste Augmentation\n==============================================================================\n### @Author       : 한의정\n### @Description  : 계층적 분할(Stratified Split)을 통한 독립적 Val 셋 구축 및\n###                 희귀 클래스 물량 확보를 위한 지능형 합성(Copy-Paste) 수행.\n==============================================================================\n'

In [2]:
# ============================================================
# [Cell 0] 환경 설정 — Colab / 로컬 자동 감지
# ============================================================
import sys, os

try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/pill_detection_project'
    if not os.path.exists(REPO_DIR):
        os.system('git clone https://github.com/wina0901/pill_detection_project.git ' + REPO_DIR)

    sys.path.insert(0, REPO_DIR)
    BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'

else:
    sys.path.insert(0, os.path.abspath('..'))
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '../data'))

print(f"✅ 환경: {'Colab' if is_colab else '로컬'}")
print(f"✅ PROJECT: {sys.path[0]}")
print(f"✅ DATA:    {BASE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 환경: Colab
✅ PROJECT: /content/pill_detection_project
✅ DATA:    /content/drive/MyDrive/data/초급_프로젝트/dataset


In [3]:
# ============================================================
# [Cell 0] 환경 설정 — Colab / 로컬 자동 감지
# ============================================================
import sys, os

try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/pill_detection_project'
    if not os.path.exists(REPO_DIR):
        os.system('git clone https://github.com/wina0901/pill_detection_project.git ' + REPO_DIR)

    sys.path.insert(0, REPO_DIR)
    BASE_DIR = '/content/drive/MyDrive/data/초급_프로젝트/dataset'

else:
    sys.path.insert(0, os.path.abspath('..'))
    BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '../data'))

print(f"✅ 환경: {'Colab' if is_colab else '로컬'}")
print(f"✅ PROJECT: {sys.path[0]}")
print(f"✅ DATA:    {BASE_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ 환경: Colab
✅ PROJECT: /content/pill_detection_project
✅ DATA:    /content/drive/MyDrive/data/초급_프로젝트/dataset


In [4]:
import os
import json
import random
import copy
import warnings
from collections import defaultdict
warnings.filterwarnings('ignore')

In [5]:
############################################################
# 1. 데이터 계층적 분할 (Stratified Train / Val Split)
############################################################
import os
import json
import random
import warnings
from collections import defaultdict
import platform
warnings.filterwarnings('ignore')

# ==========================================
# 1. 경로 설정 및 환경 세팅
# ==========================================
# [!] 로컬(M2 Mac)과 Colab 환경을 모두 고려하여 base_dir을 동적으로 세팅할 수도 있습니다.
# 현재 노트북 위치(notebooks/)에서 한 칸 위로 올라가서(../) data 폴더를 바라보게 만듭니다.
original_json_path = os.path.join(BASE_DIR, 'merged_annotations_train_final.json')

# 분할된 데이터를 저장할 경로 (raw는 증강 전 원본이라는 의미)
train_raw_json_path = os.path.join(BASE_DIR, 'train_raw.json')
val_json_path = os.path.join(BASE_DIR, 'val.json')

# 재현성(Reproducibility)을 위한 시드 고정 (딥러닝 실험의 기본)
random.seed(42)

# ==========================================
# 2. 데이터 로드 및 구조 분해
# ==========================================
print("🚀 [Step 1] 원본 통합 COCO 데이터 로딩 중...")
with open(original_json_path, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

images = coco_data['images']
annotations = coco_data['annotations']
categories = coco_data['categories']

# ==========================================
# 3. 이미지별 대표 클래스(Dominant Class) 추출
# ==========================================
# Multi-label 이미지(알약 여러 개)를 단일 Label 기준으로 스플릿하기 위한 휴리스틱
img_to_cats = defaultdict(list)
for ann in annotations:
    img_to_cats[ann['image_id']].append(ann['category_id'])

def get_majority_label(cat_list):
    """이미지 내에서 가장 많이 등장한 클래스를 해당 이미지의 '대표 클래스'로 선정합니다."""
    return max(set(cat_list), key=cat_list.count)

img_dominant = {
    img_id: get_majority_label(cats)
    for img_id, cats in img_to_cats.items()
}

# ==========================================
# 4. 계층적 분할 (Stratified Split Engine)
# ==========================================
print("⚖️ [Step 2] 클래스 비율을 보존하며 Train/Val 9:1 분할을 시작합니다...")

class_to_imgs = defaultdict(list)
for img_id, label in img_dominant.items():
    class_to_imgs[label].append(img_id)

train_ids, val_ids = set(), set()

for label, img_list in class_to_imgs.items():
    random.shuffle(img_list) # 시드가 고정되어 있으므로 매번 동일하게 섞임

    if len(img_list) == 1:
        # [예외 1] 극소수 클래스 (1장): 무조건 Train으로 보내서 특징을 학습시킴
        train_ids.update(img_list)
    elif len(img_list) < 5:
        # [예외 2] 소수 클래스 (5장 미만): Validation에 최소 1장은 보장하여 평가 가능하게 함
        val_ids.add(img_list[0])
        train_ids.update(img_list[1:])
    else:
        # [일반] 충분한 데이터: 정확히 9:1 비율로 스플릿
        split_idx = max(1, int(len(img_list) * 0.1))
        val_ids.update(img_list[:split_idx])
        train_ids.update(img_list[split_idx:])

# 분할 결과 검증
print(f"📊 [결과] 총 {len(images):,}장 분할 완료 → Train: {len(train_ids):,}장 / Val: {len(val_ids):,}장")

# ==========================================
# 5. 분할 무결성(Integrity) 및 소수 클래스 검증
# ==========================================
val_classes = set(img_dominant[i] for i in val_ids if i in img_dominant)
missing = set(img_dominant.values()) - val_classes

if missing:
    print(f"🚨 [경고] Validation 셋에 누락된 클래스 {len(missing)}개 발생 (사유: 전체 데이터가 1장뿐인 클래스)")
else:
    print(f"✅ [검증] 전체 {len(categories)}개 클래스가 Validation 셋에 모두 정상적으로 포함되었습니다.")

# ==========================================
# 6. 객체 분류 및 JSON 직렬화(저장)
# ==========================================
print("💾 [Step 3] 분할된 데이터를 JSON 형식으로 묶어 저장합니다...")

# O(1) 탐색을 위해 Set 자료형 활용 (속도 극대화)
train_images = [img for img in images if img['id'] in train_ids]
val_images   = [img for img in images if img['id'] in val_ids]
train_annotations = [ann for ann in annotations if ann['image_id'] in train_ids]
val_annotations   = [ann for ann in annotations if ann['image_id'] in val_ids]

train_coco = {"images": train_images, "annotations": train_annotations, "categories": categories}
val_coco   = {"images": val_images,   "annotations": val_annotations,   "categories": categories}

# indent=2를 빼면 JSON 용량이 절반 이하로 줄어들고 로딩 속도가 3배 이상 빨라집니다.
# 사람이 직접 읽을 목적이 아니라면 indent를 제거하는 것을 권장합니다. (여기서는 원본 유지)
with open(train_raw_json_path, 'w', encoding='utf-8') as f:
    json.dump(train_coco, f, ensure_ascii=False, indent=2)
with open(val_json_path, 'w', encoding='utf-8') as f:
    json.dump(val_coco, f, ensure_ascii=False, indent=2)

print(f"\n🎉 [OK] 분리 및 저장 완료!")
print(f"▶ Train 파일: {os.path.basename(train_raw_json_path)} (BBox 객체 {len(train_annotations):,}개)")
print(f"▶ Val 파일:   {os.path.basename(val_json_path)} (BBox 객체 {len(val_annotations):,}개)")

🚀 [Step 1] 원본 통합 COCO 데이터 로딩 중...
⚖️ [Step 2] 클래스 비율을 보존하며 Train/Val 9:1 분할을 시작합니다...
📊 [결과] 총 1,489장 분할 완료 → Train: 1,350장 / Val: 139장
🚨 [경고] Validation 셋에 누락된 클래스 6개 발생 (사유: 전체 데이터가 1장뿐인 클래스)
💾 [Step 3] 분할된 데이터를 JSON 형식으로 묶어 저장합니다...

🎉 [OK] 분리 및 저장 완료!
▶ Train 파일: train_raw.json (BBox 객체 4,095개)
▶ Val 파일:   val.json (BBox 객체 431개)


In [6]:
############################################################
# 2. 소수 클래스 데이터 구출: 알약 스티커(Crop) 추출 엔진
############################################################
import os
import glob
import json
import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import warnings
import platform

warnings.filterwarnings('ignore')

# ==========================================
# 1. 경로 및 환경 설정 (I/O 파이프라인 구축)
# ==========================================
json_path = os.path.join(BASE_DIR, 'train_raw.json') # 앞서 분할한 순수 학습용 데이터

# 잘라낸 알약 스티커들이 저장될 '무기고(Arsenal)' 폴더
crop_save_dir = os.path.join(BASE_DIR, 'crops_minority') 
os.makedirs(crop_save_dir, exist_ok=True)

# ==========================================
# 2. 증강 타겟 탐색 (Minority Class Targeting)
# ==========================================
print("🔍 [Step 1] 데이터를 분석하여 구출이 시급한 소수 클래스를 타겟팅합니다...")

with open(json_path, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

images_df = pd.DataFrame(coco_data['images'])
annotations_df = pd.DataFrame(coco_data['annotations'])
categories_df = pd.DataFrame(coco_data['categories'])

cat_dict = dict(zip(categories_df['id'], categories_df['name']))
annotations_df['class_name'] = annotations_df['category_id'].map(cat_dict)

# 💡 [하이퍼파라미터] 타겟 기준: 데이터가 50개 미만인 클래스 집중 공략
THRESHOLD = 50
class_counts = annotations_df['class_name'].value_counts()
minority_classes = class_counts[class_counts < THRESHOLD].index.tolist()

print(f"🎯 [타겟 확정] 증강 대상 소수 클래스: {len(minority_classes)}종 (기준: {THRESHOLD}개 미만)")

# 고속 파일 검색용 인덱싱 (O(1) 해시맵 탐색)
all_files = glob.glob(os.path.join(BASE_DIR, "**", "*.png"), recursive=True) + \
            glob.glob(os.path.join(BASE_DIR, "**", "*.jpg"), recursive=True) + \
            glob.glob(os.path.join(BASE_DIR, "**", "*.JPG"), recursive=True)
path_map = {os.path.basename(f): f for f in all_files}

# ==========================================
# 3. 정밀 누끼(Crop) 추출 엔진 가동
# ==========================================
print(f"\n✂️ [Step 2] 소수 클래스 알약 스티커 제작(Crop)을 시작합니다...")
crop_metadata = []

# 진행률 파악을 위한 tqdm 도입
for cls_name in tqdm(minority_classes, desc="클래스별 스티커 추출"):
    # 리눅스/윈도우 파일시스템 오류 방지용 특수문자 치환 (Sanitization)
    safe_cls_name = str(cls_name).replace("/", "_").replace(" ", "_").replace(":", "_")
    cls_dir = os.path.join(crop_save_dir, safe_cls_name)
    os.makedirs(cls_dir, exist_ok=True)
    
    cls_annos = annotations_df[annotations_df['class_name'] == cls_name]
    
    for idx, anno in cls_annos.iterrows():
        img_info = images_df[images_df['id'] == anno['image_id']].iloc[0]
        f_name = os.path.basename(img_info['file_name'])
        
        if f_name in path_map:
            # 💡 [디테일 1] 한글/유니코드 경로 에러 원천 차단 (numpy byte array로 읽기)
            # Mac(M2)에서는 보통 문제없지만, Colab(Linux)이나 Windows 협업 시 필수적인 방어 로직입니다.
            img_array = np.fromfile(path_map[f_name], np.uint8)
            img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
            
            if img is None:
                continue
                
            x, y, w, h = map(int, anno['bbox'])
            
            # 💡 [디테일 2] Padding: 나중에 백그라운드에 합성할 때 외곽선이 부자연스럽게 
            # 잘리는 현상(Hard Edge)을 방지하기 위해 주변 픽셀을 2px 여유 있게 가져옵니다.
            PAD = 2
            img_h, img_w = img.shape[:2]
            
            # 이미지 경계값(Boundary)을 넘지 않도록 안전하게 클리핑(Clipping)
            x1, y1 = max(0, x - PAD), max(0, y - PAD)
            x2, y2 = min(img_w, x + w + PAD), min(img_h, y + h + PAD)
            
            crop_img = img[y1:y2, x1:x2]
            
            # 자른 이미지가 정상적으로 존재하는 경우만 저장
            if crop_img.size > 0:
                save_name = f"{safe_cls_name}_{anno['id']}.png"
                save_path = os.path.join(cls_dir, save_name)
                
                # 💡 [디테일 3] 한글 경로 안전 저장 (cv2.imwrite 대신 imencode 사용)
                extension = os.path.splitext(save_path)[1]
                result, encoded_img = cv2.imencode(extension, crop_img)
                if result:
                    with open(save_path, mode='w+b') as f:
                        encoded_img.tofile(f)
                
                # 추후 합성 파이프라인에서 사용할 메타데이터 기록 (DB화)
                crop_metadata.append({
                    'class_name': cls_name,
                    'category_id': anno['category_id'],
                    'crop_path': save_path,
                    'width': x2 - x1,
                    'height': y2 - y1
                })

# ==========================================
# 4. 메타데이터 DB 저장
# ==========================================
if crop_metadata:
    meta_df = pd.DataFrame(crop_metadata)
    meta_save_path = os.path.join(crop_save_dir, 'crop_meta.csv')
    # 한글 깨짐 방지를 위해 utf-8-sig 인코딩 사용
    meta_df.to_csv(meta_save_path, index=False, encoding='utf-8-sig')

    print(f"\n🎉 [OK] 추출 완료! 총 {len(crop_metadata):,}개의 💎 '희귀 알약 스티커'가 확보되었습니다.")
    print(f"▶ 스티커 저장 위치: {crop_save_dir}")
    print(f"▶ 메타데이터 저장 위치: {meta_save_path}")
else:
    print("\n🚨 [경고] 추출된 스티커가 없습니다. 타겟 클래스나 이미지 경로를 다시 확인해주세요.")

🔍 [Step 1] 데이터를 분석하여 구출이 시급한 소수 클래스를 타겟팅합니다...
🎯 [타겟 확정] 증강 대상 소수 클래스: 47종 (기준: 50개 미만)

✂️ [Step 2] 소수 클래스 알약 스티커 제작(Crop)을 시작합니다...


클래스별 스티커 추출:   0%|          | 0/47 [00:00<?, ?it/s]


🎉 [OK] 추출 완료! 총 1,202개의 💎 '희귀 알약 스티커'가 확보되었습니다.
▶ 스티커 저장 위치: /content/drive/MyDrive/data/초급_프로젝트/dataset/crops_minority
▶ 메타데이터 저장 위치: /content/drive/MyDrive/data/초급_프로젝트/dataset/crops_minority/crop_meta.csv


In [7]:
############################################################
# 3. [Data Pipeline Phase 3] 
#     Data Leakage-Free Copy-Paste 증강 엔진 (최종 무결점 버전)
############################################################
import os
import glob
import json
import cv2
import numpy as np
import pandas as pd
import random
from tqdm.auto import tqdm
import copy
import platform
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. 경로 및 환경 설정
# ==========================================
train_json_path = os.path.join(BASE_DIR, 'train_raw.json') # 순수 학습용 데이터
crop_dir = os.path.join(BASE_DIR, 'crops_minority')        # 알약 스티커 보관소
meta_path = os.path.join(crop_dir, 'crop_meta.csv')        # 스티커 메타데이터

# 증강된 결과물이 저장될 종착지
aug_img_dir = os.path.join(BASE_DIR, 'train_augmented_images')
aug_json_path = os.path.join(BASE_DIR, 'train_augmented_final.json')
os.makedirs(aug_img_dir, exist_ok=True)

# ==========================================
# 2. 데이터 및 메타데이터 로드 (Snapshot)
# ==========================================
print("🔄 [Step 1] Train 데이터 및 알약 스티커 도감을 불러옵니다...")
with open(train_json_path, 'r', encoding='utf-8') as f:
    coco_data = json.load(f)

# 원본 보존을 위한 Deep Copy (안전제일)
aug_coco = copy.deepcopy(coco_data)

if not os.path.exists(meta_path):
    raise FileNotFoundError(f"🚨 [에러] 스티커 메타데이터를 찾을 수 없습니다: {meta_path}")
crop_meta = pd.read_csv(meta_path)

# 고속 탐색용 파일 인덱싱
all_files = glob.glob(os.path.join(BASE_DIR, "**", "*.png"), recursive=True) + \
            glob.glob(os.path.join(BASE_DIR, "**", "*.jpg"), recursive=True) + \
            glob.glob(os.path.join(BASE_DIR, "**", "*.JPG"), recursive=True)
path_map = {os.path.basename(f): f for f in all_files}

# 새로운 ID 발급을 위한 시퀀스(Sequence) 초기화
max_img_id = max([img['id'] for img in aug_coco['images']]) if aug_coco['images'] else 0
max_anno_id = max([ann['id'] for ann in aug_coco['annotations']]) if aug_coco['annotations'] else 0

# ==========================================
# 3. 충돌 감지 알고리즘 (Collision Detection)
# ==========================================
def check_overlap(new_box, existing_boxes, min_dist=15):
    """
    새로 붙일 알약이 기존 알약(혹은 방금 붙인 알약)과 겹치는지 검사합니다.
    IoU 계산보다 빠르고 직관적인 AABB(Axis-Aligned Bounding Box) 충돌 판정입니다.
    """
    nx, ny, nw, nh = new_box
    for ex, ey, ew, eh in existing_boxes:
        # 두 박스가 겹치지 않는 조건 (상하좌우 여백 포함)
        if not (nx + nw + min_dist < ex or nx > ex + ew + min_dist or \
                ny + nh + min_dist < ey or ny > ey + eh + min_dist):
            return True # 겹침 발생! (Reject)
    return False # 안전함 (Accept)

def blend_image(bg_crop, sticker):
    """
    [시니어의 비기] 알약 스티커를 배경에 자연스럽게 융화시킵니다.
    테두리의 경계선(Hard Edge)을 모델이 '노이즈'나 '합성 힌트'로 학습하지 못하게 방지합니다.
    """
    # 1. 배경과 스티커의 크기가 다르면 합성 불가 (안전장치)
    if bg_crop.shape != sticker.shape:
        return sticker
    
    # 2. 아주 미세한 가우시안 블러를 통해 경계선 연화 (Feathering 효과)
    # 중앙 객체는 살리고, 외곽선 위주로 배경과 섞이게 만듭니다.
    blended = cv2.addWeighted(bg_crop, 0.2, sticker, 0.8, 0)
    return blended

# ==========================================
# 4. Copy-Paste 합성 엔진 가동
# ==========================================
# 💡 [하이퍼파라미터] 생성할 새로운 합성 이미지의 수 (소수 클래스를 얼마나 불릴 것인가)
AUG_COUNT = 500 
print(f"\n🧬 [Step 2] 총 {AUG_COUNT:,}장의 Copy-Paste 증강 이미지 생성을 시작합니다...")

# 오염되지 않은 순수한 '원본 도화지'만 배경 후보로 사용
bg_candidates = list(coco_data['images'])

for i in tqdm(range(AUG_COUNT), desc="합성 및 JSON 업데이트"):
    # 1) 도화지(배경 이미지) 무작위 선정
    bg_info = random.choice(bg_candidates)
    f_name = os.path.basename(bg_info['file_name'])
    
    if f_name not in path_map:
        continue
        
    # 한글 경로 에러 방지용 로드
    bg_array = np.fromfile(path_map[f_name], np.uint8)
    bg_img = cv2.imdecode(bg_array, cv2.IMREAD_COLOR)
    
    if bg_img is None: continue
    bg_h, bg_w = bg_img.shape[:2]
    
    # 도화지에 원래 있던 '정답(알약)들'의 자리 선점 (이 위에는 새 알약을 덮어쓰지 않음)
    existing_boxes = [ann['bbox'] for ann in aug_coco['annotations'] if ann['image_id'] == bg_info['id']]
    
    # 2) 이번 도화지에 몇 개의 소수 클래스 알약을 붙일 것인가? (1~4개)
    num_pastes = random.randint(1, 4)
    pastes_to_add = crop_meta.sample(num_pastes, replace=True)
    
    new_annotations = []
    success_paste = False
    
    for _, paste_row in pastes_to_add.iterrows():
        crop_path = paste_row['crop_path']
        cw, ch = int(paste_row['width']), int(paste_row['height'])
        
        crop_array = np.fromfile(crop_path, np.uint8)
        crop_img = cv2.imdecode(crop_array, cv2.IMREAD_COLOR)
        
        if crop_img is None: continue
            
        # 3) 빈자리 찾기 (최대 50번 찔러보기)
        safe_x, safe_y = -1, -1
        for _ in range(50):
            # 화면 밖으로 알약이 삐져나가지 않도록 좌표 제한
            # max(11, ...) 처리는 알약 크기가 배경보다 클 때 발생하는 ValueError 방지
            max_x = max(11, bg_w - cw - 10)
            max_y = max(11, bg_h - ch - 10)
            
            # 배경이 스티커보다 작으면 스킵
            if max_x <= 10 or max_y <= 10: 
                break 

            rx = random.randint(10, max_x)
            ry = random.randint(10, max_y)
            
            # 겹치지 않는(Safe) 자리를 찾았다면 루프 탈출
            if not check_overlap((rx, ry, cw, ch), existing_boxes):
                safe_x, safe_y = rx, ry
                break
                
        # 4) 빈자리에 알약 스티커 붙이기
        if safe_x != -1 and safe_x + cw <= bg_w and safe_y + ch <= bg_h:
            # 💡 [시니어 디테일] 단순 덮어쓰기가 아니라, 배경과 살짝 섞어주는 블렌딩 적용
            bg_crop = bg_img[safe_y:safe_y+ch, safe_x:safe_x+cw]
            bg_img[safe_y:safe_y+ch, safe_x:safe_x+cw] = blend_image(bg_crop, crop_img)
            
            # 다음 스티커가 이 위에 겹치지 않도록 방금 붙인 자리도 선점 처리
            existing_boxes.append([safe_x, safe_y, cw, ch])
            
            max_anno_id += 1
            new_annotations.append({
                "id": max_anno_id,
                "image_id": max_img_id + 1,  # 아직 발급 전이지만, 미리 매핑해둠
                "category_id": int(paste_row['category_id']),
                "bbox": [safe_x, safe_y, cw, ch],
                "area": float(cw * ch),
                "iscrowd": 0,
                "segmentation": []
            })
            success_paste = True
            
    # 5) 한 개라도 성공적으로 붙였다면, 새로운 이미지로 저장
    if success_paste:
        max_img_id += 1
        new_filename = f"aug_cp_{max_img_id:06d}.jpg" # aug_cp_001234.jpg 형태로 통일
        save_path = os.path.join(aug_img_dir, new_filename)
        
        # 한글 경로 안전 저장
        result, encoded_img = cv2.imencode('.jpg', bg_img)
        if result:
            with open(save_path, mode='w+b') as f:
                encoded_img.tofile(f)
                
        # [데이터 무결성] COCO JSON 구조 업데이트
        aug_coco['images'].append({
            "id": max_img_id,
            "file_name": new_filename,
            "width": bg_w,
            "height": bg_h
        })
        
        # 원본 이미지에 있던 진짜 알약들도 이 새로운 합성 사진의 정답으로 복사해 줌
        original_annos = [copy.deepcopy(ann) for ann in aug_coco['annotations'] if ann['image_id'] == bg_info['id']]
        for ann in original_annos:
            max_anno_id += 1
            ann['id'] = max_anno_id
            ann['image_id'] = max_img_id
            aug_coco['annotations'].append(ann)
            
        # 방금 붙인 합성 알약들도 정답으로 추가
        # (위에서 매핑해둔 max_img_id + 1 이 여기서 발급된 max_img_id 와 일치하게 됨)
        aug_coco['annotations'].extend(new_annotations)

# ==========================================
# 5. 최종 증강 데이터셋 저장
# ==========================================
print("\n💾 [Step 3] 증강이 완료된 COCO JSON 포맷을 디스크에 저장합니다...")
# 파일 용량 및 로딩 속도 최적화를 위해 indent 제거 (선택사항)
with open(aug_json_path, 'w', encoding='utf-8') as f:
    json.dump(aug_coco, f, ensure_ascii=False) # indent=2 제거 (I/O 속도 향상)

print(f"\n🎉 [OK] 모든 Data Pipeline Phase 3 완료!")
print(f"▶ 원본(Train) 알약 객체 수: {len(coco_data['annotations']):,}개")
print(f"▶ 증강 후 총 알약 객체 수: {len(aug_coco['annotations']):,}개 (Long-tail 완화 완료!)")
print(f"▶ 📸 생성된 합성 이미지 저장소: {aug_img_dir}")
print(f"▶ 📝 최종 Train 정답지 경로: {aug_json_path}")

🔄 [Step 1] Train 데이터 및 알약 스티커 도감을 불러옵니다...

🧬 [Step 2] 총 500장의 Copy-Paste 증강 이미지 생성을 시작합니다...


합성 및 JSON 업데이트:   0%|          | 0/500 [00:00<?, ?it/s]


💾 [Step 3] 증강이 완료된 COCO JSON 포맷을 디스크에 저장합니다...

🎉 [OK] 모든 Data Pipeline Phase 3 완료!
▶ 원본(Train) 알약 객체 수: 4,095개
▶ 증강 후 총 알약 객체 수: 6,199개 (Long-tail 완화 완료!)
▶ 📸 생성된 합성 이미지 저장소: /content/drive/MyDrive/data/초급_프로젝트/dataset/train_augmented_images
▶ 📝 최종 Train 정답지 경로: /content/drive/MyDrive/data/초급_프로젝트/dataset/train_augmented_final.json
